In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MyHealthcareApp").getOrCreate()

In [0]:
insurance_file_path = r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/bronze/dbo.insurance.csv'
insurance_df = spark.read.csv(insurance_file_path, header=True, inferSchema=True)
insurance_df.count()

In [0]:
#Deduplication
from pyspark.sql import functions as F

insurance_insurance_id_df = insurance_df.dropDuplicates(
    ["insurance_id"]
)

# insurance_insurance_id_df.groupBy(
#     "insurance_id"
# ).count().filter(
#     F.col("count") > 1
# ).show(truncate=False)

In [0]:
#Casing Standardization
from pyspark.sql import functions as F

insurance_insurer_name_std_df = insurance_insurance_id_df.withColumn(
    "insurer_name_std",
    F.when(
        F.col("insurer_name").isNull(),
        "Unknown"
    ).otherwise(
        F.initcap(
            F.trim(
                F.col("insurer_name")
            )
        )
    )
)

# insurance_insurer_name_std_df.select(
#     "insurer_name",
#     "insurer_name_std"
# ).show(20, truncate=False)

In [0]:
#Text Cleansing
from pyspark.sql import functions as F

insurance_policy_number_clean_df = insurance_insurer_name_std_df.withColumn(
    "policy_number_clean",
    F.upper(
        F.trim(
            F.col("policy_number")
        )
    )
).withColumn(
    "dq_flag",
    F.when(
        F.col("policy_number").isNull(),
        1
    ).otherwise(0)
)

# insurance_policy_number_clean_df.select(
#     "policy_number",
#     "policy_number_clean",
#     "dq_flag"
# ).show(20, truncate=False)

In [0]:
#Casing Standardization
from pyspark.sql import functions as F

insurance_plan_name_std_df = insurance_df.withColumn(
    "plan_name_std",
    F.initcap(
        F.trim(
            F.col("plan_name")
        )
    )
)
# insurance_plan_name_std_df.select(
#     "plan_name",
#     "plan_name_std"
# ).show(20, truncate=False)

In [0]:
#Standardization
from pyspark.sql import functions as F

insurance_coverage_type_std_df = insurance_plan_name_std_df.withColumn(
    "coverage_type_std",
    F.when(
        F.col("coverage_type").isNull(),
        "Unknown"
    ).otherwise(
        F.initcap(
            F.trim(
                F.col("coverage_type")
            )
        )
    )
)

# insurance_coverage_type_std_df.select(
#     "coverage_type",
#     "coverage_type_std"
# ).show(20, truncate=False)

In [0]:
#Range Validation
from pyspark.sql import functions as F

insurance_sum_insured_valid_df = insurance_coverage_type_std_df.withColumn(
    "sum_insured_valid",
    F.when(
        F.col("sum_insured") > 0,
        F.col("sum_insured").cast("double")
    ).otherwise(None)
)

# insurance_sum_insured_valid_df.select(
#     "sum_insured",
#     "sum_insured_valid"
# ).show(20, truncate=False)

In [0]:
#Range Validation
from pyspark.sql import functions as F

insurance_deductible_valid_df = insurance_sum_insured_valid_df.withColumn(
    "deductible_valid",
    F.when(F.col("deductible") >= 0, F.col("deductible").cast("double"))
     .otherwise(None)
)

# insurance_deductible_valid_df.filter(
#     F.expr("""
#         NOT (
#             (deductible >= 0 AND deductible_valid = CAST(deductible AS DOUBLE)) OR
#             (deductible < 0 AND deductible_valid IS NULL)
#         )
#     """)
# ).show()

In [0]:
#Range Validation
from pyspark.sql import functions as F

insurance_copay_pct_valid_df = insurance_deductible_valid_df.withColumn(
    "copay_pct_valid",
    F.when(
        (F.col("copay_percent") >= 0) & (F.col("copay_percent") <= 100),
        F.col("copay_percent").cast("double")
    ).otherwise(None)
)

# insurance_copay_pct_valid_df.filter(
#     F.expr("""
#         NOT (
#             (copay_percent BETWEEN 0 AND 100 
#                 AND copay_pct_valid = CAST(copay_percent AS DOUBLE)) 
#             OR
#             (copay_percent < 0 OR copay_percent > 100 
#                 AND copay_pct_valid IS NULL)
#         )
#     """)
# ).show()

In [0]:
#Date Standardization
from pyspark.sql import functions as F

insurance_policy_start_std_df = insurance_copay_pct_valid_df.withColumn(
    "policy_start_std",
    F.coalesce(
        F.try_to_date(F.col("policy_start_date"), "dd-MM-yyyy"),
        F.try_to_date(F.col("policy_start_date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("policy_start_date"), "MM/dd/yyyy"),
        F.try_to_date(F.col("policy_start_date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("policy_start_date"), "MM-dd-yy"),
        F.try_to_date(F.col("policy_start_date"), "dd-MM-yy"),
        F.try_to_date(F.col("policy_start_date"), "MM/dd/yy"),
        F.try_to_date(F.col("policy_start_date"), "dd/MM/yy")
    )
)

# insurance_policy_start_std_df.filter(
#     F.expr("""
#         NOT (
#             (
#                 policy_start_date RLIKE '^[0-9]{2}-[0-9]{2}-[0-9]{4}$' OR
#                 policy_start_date RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$' OR
#                 policy_start_date RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$'
#             )
#             AND policy_start_std IS NOT NULL
#         )
#         AND policy_start_std IS NULL
#     """)
# ).show()

In [0]:
#Date Standardization
from pyspark.sql import functions as F

insurance_policy_end_std_df = insurance_policy_start_std_df.withColumn(
    "policy_end_std",
    F.coalesce(
        F.try_to_date(F.col("policy_end_date"), "dd-MM-yyyy"),
        F.try_to_date(F.col("policy_end_date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("policy_end_date"), "MM/dd/yyyy"),
        F.try_to_date(F.col("policy_end_date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("policy_end_date"), "MM-dd-yy"),
        F.try_to_date(F.col("policy_end_date"), "dd-MM-yy"),
        F.try_to_date(F.col("policy_end_date"), "MM/dd/yy"),
        F.try_to_date(F.col("policy_end_date"), "dd/MM/yy")
    )
)

# insurance_policy_end_std_df.filter(
#     F.expr("""
#         NOT (
#             (
#                 policy_end_date RLIKE '^[0-9]{2}-[0-9]{2}-[0-9]{4}$' OR
#                 policy_end_date RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$' OR
#                 policy_end_date RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$'
#             )
#             AND policy_end_std IS NOT NULL
#         )
#         AND policy_end_std IS NULL
#     """)
# ).show()

In [0]:
#Derived – Tenure
from pyspark.sql import functions as F

insurance_policy_tenure_days_df = insurance_policy_end_std_df.withColumn(
    "policy_tenure_days",
    F.datediff(F.col("policy_end_std"), F.col("policy_start_std"))
)

# insurance_policy_tenure_days_df.select(
#     "policy_start_std",
#     "policy_end_std",
#     "policy_tenure_days",
#     F.datediff(
#         F.col("policy_end_std"),
#         F.col("policy_start_std")
#     ).alias("expected_value")
# ).show()

In [0]:
#Derived Flag
from pyspark.sql import functions as F

insurance_is_policy_expired_df = insurance_policy_tenure_days_df.withColumn(
    "is_policy_expired",
    F.when(
        F.col("policy_end_std").isNull(),
        None
    ).when(
        F.col("policy_end_std") < F.current_date(),
        F.lit(1)
    ).otherwise(F.lit(0))
)

# insurance_is_policy_expired_df.select(
#     "policy_end_std",
#     "is_policy_expired",
#     F.when(
#         F.col("policy_end_std").isNull(),
#         None
#     ).when(
#         F.col("policy_end_std") < F.current_date(),
#         F.lit(1)
#     ).otherwise(F.lit(0)).alias("expected_value")
# ).show()

In [0]:
#Casing Standardization
from pyspark.sql import functions as F

insurance_tpa_name_std_df = insurance_is_policy_expired_df.withColumn(
    "tpa_name_std",
    F.when(
        F.col("tpa_name").isNull(),
        None
    ).otherwise(
        F.initcap(F.trim(F.col("tpa_name")))
    )
)

# insurance_tpa_name_std_df.select(
#     "tpa_name",
#     "tpa_name_std",
#     F.when(
#         F.col("tpa_name").isNull(),
#         None
#     ).otherwise(
#         F.initcap(F.trim(F.col("tpa_name")))
#     ).alias("expected_value")
# ).show()

In [0]:
#Boolean Normalisation
from pyspark.sql import functions as F

from pyspark.sql import functions as F

insurance_is_active_flag_df = insurance_tpa_name_std_df.withColumn(
    "is_active_flag",
    F.when(F.col("is_active").isNull(), F.lit(0))
     .when(F.upper(F.col("is_active")).isin("Y", "YES", "1"), F.lit(1))
     .when(F.upper(F.col("is_active")).isin("N", "NO", "0"), F.lit(0))
     .otherwise(None)
)

# insurance_is_active_flag_df.select(
#     "is_active",
#     "is_active_flag",
#     F.when(F.col("is_active").isNull(), F.lit(0))
#      .when(F.upper(F.col("is_active")).isin("Y", "YES", "1"), F.lit(1))
#      .when(F.upper(F.col("is_active")).isin("N", "NO", "0"), F.lit(0))
#      .otherwise(None)
#      .alias("expected_value")
# ).show()

In [0]:
#Derived – Tier
from pyspark.sql import functions as F

insurance_coverage_tier_df = insurance_is_active_flag_df.withColumn(
    "coverage_tier",
    F.when(F.col("sum_insured_valid").isNull(), F.lit("Unknown"))
     .when(F.col("sum_insured_valid") < 500000, F.lit("Basic"))
     .when((F.col("sum_insured_valid") >= 500000) & (F.col("sum_insured_valid") < 1000000), F.lit("Standard"))
     .when((F.col("sum_insured_valid") >= 1000000) & (F.col("sum_insured_valid") < 2000000), F.lit("Enhanced"))
     .otherwise(F.lit("Premium"))
)

# insurance_coverage_tier_df.select(
#     "sum_insured_valid",
#     "coverage_tier",
#     F.when(F.col("sum_insured_valid").isNull(), F.lit("Unknown"))
#      .when(F.col("sum_insured_valid") < 500000, F.lit("Basic"))
#      .when((F.col("sum_insured_valid") >= 500000) & (F.col("sum_insured_valid") < 1000000), F.lit("Standard"))
#      .when((F.col("sum_insured_valid") >= 1000000) & (F.col("sum_insured_valid") < 2000000), F.lit("Enhanced"))
#      .otherwise(F.lit("Premium"))
#      .alias("expected_value")
# ).show()

In [0]:
#Data Quality Flag
from pyspark.sql import functions as F

insurance_dq_insurance_flag_df = insurance_coverage_tier_df.withColumn(
    "dq_insurance_flag",
    F.when(
        F.col("policy_number").isNull() |
        F.col("sum_insured_valid").isNull() |
        F.col("policy_start_std").isNull(),
        F.lit(1)
    ).otherwise(F.lit(0))
)
insurance_dq_insurance_flag_df.printSchema()
display(insurance_dq_insurance_flag_df)
# insurance_dq_insurance_flag_df.select(
#     "policy_number",
#     "sum_insured_valid",
#     "policy_start_std",
#     "dq_insurance_flag",
#     F.when(
#         F.col("policy_number").isNull() |
#         F.col("sum_insured_valid").isNull() |
#         F.col("policy_start_std").isNull(),
#         F.lit(1)
#     ).otherwise(F.lit(0)).alias("expected_value")
# ).show()

In [0]:
# #Derived – Date Extract
# from pyspark.sql import functions as F

# # insurance_ingestion_date_df = insurance_coverage_tier_df.withColumn(
# #     "ingestion_date",
# #     F.to_date(F.col("ingestion_timestamp"))
# # )

# #If ingestion_timestamp is string type:
# insurance_ingestion_date_df = insurance_dq_insurance_flag_df.withColumn(
#     "ingestion_date",
#     F.to_date(
#         F.to_timestamp(F.col("ingestion_timestamp"))
#     )
# )

# # df_check = insurance_ingestion_date_df.select(
# #     "ingestion_timestamp",
# #     "ingestion_date",
# #     F.to_date(F.col("ingestion_timestamp")).alias("expected_value")
# # )

# # df_check.show()

# insurance_ingestion_date_df.filter(
#     F.expr("""
#         try_cast(ingestion_timestamp as timestamp) IS NULL
#         AND ingestion_timestamp IS NOT NULL
#     """)
# ).select("ingestion_timestamp").count()


In [0]:
# #FK Validation
# from pyspark.sql import functions as F

# # Load reference table
# patients_df = spark.table("patients_silver").select("patient_id").distinct()

# insurance_patient_id_clean_df = insurance_coverage_tier_df.withColumn(
#     "patient_id_clean",
#     F.trim(F.col("patient_id"))
# )

# insurance_patient_id_valid_df = insurance_patient_id_clean_df.join(
#     patients_df,
#     insurance_patient_id_clean_df.patient_id_clean == patients_df.patient_id,
#     "left"
# ).withColumn(
#     "patient_id_valid",
#     F.when(F.col("patients_silver.patient_id").isNotNull(),
#            F.col("patient_id_clean"))
#      .otherwise(None)
# )

# insurance_patient_id_valid_df.join(
#     spark.table("patients_silver").select("patient_id"),
#     df.patient_id_valid == F.col("patient_id"),
#     "left_anti"
# ).show()

In [0]:
# ----------------------------------------
#  Write to Silver layer in delta format
# ----------------------------------------


TABLE_NAME = "insurance"  

silver_table_path = f"abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/silver/{TABLE_NAME}"

# ── OVERWRITE (Full Load) ────────────────────────────────────
insurance_dq_insurance_flag_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_table_path)

print(f"✅ Data written to Silver layer: {silver_table_path}")